[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/36.%20The%20Berth%20%26%20Quay%20Length%20Design%20Problem/P36-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 36. The Berth & Quay Length Design Problem

## Tier 2: The Classic Heuristic (Divide and Conquer Decomposition)

### Goal
Implement a divide and conquer algorithm that decomposes the complex berth design problem into manageable subproblems, providing efficient solutions while maintaining solution quality.

### Key assumptions
- The problem can be decomposed into three interconnected subproblems
- Total capacity requirements depend primarily on long-term traffic forecasts
- Berth segmentation depends on vessel size distributions and operational flexibility
- Allocation verification ensures feasibility of the integrated solution
- Subproblem solutions can be effectively coordinated through interface variables

### Approach (step-by-step)
1. Implement the three-phase divide and conquer framework
2. Phase 1: Solve capacity sizing subproblem for total quay length requirements
3. Phase 2: Solve berth segmentation subproblem for optimal number and sizes of berths
4. Phase 3: Solve allocation verification subproblem to validate feasibility
5. Integrate subproblem solutions and analyze final results

### What to look for in the results
- Total quay length requirement from capacity sizing phase
- Optimal berth configuration from segmentation phase
- Feasibility verification results from allocation phase
- Computational efficiency compared to integrated optimization
- Solution quality comparison with Tier 1 robust optimization

### Concrete example (from the source)
Port with vessel data:
- Arrival rates: [15, 12, 8, 4] vessels per week
- Vessel lengths: [180, 240, 320, 420] meters
- Service rates: [8, 6, 4, 3] vessels per week
- Maximum total length: 1800 meters
- Minimum berth length: 200 meters
- Construction cost: $25,000 per meter
- Waiting cost: $8,000 per hour

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages for divide and conquer algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

In [2]:
# Define data structures for divide and conquer approach
from dataclasses import dataclass
from typing import List, Dict, Tuple

@dataclass
class VesselType:
    """Represents a vessel type with its characteristics"""
    id: str
    length: float  # meters
    arrival_rate: float  # vessels per week
    service_rate: float  # vessels per week
    
@dataclass
class CapacitySolution:
    """Solution from capacity sizing subproblem"""
    total_quay_length: float  # meters
    total_capacity: float  # vessels per week
    utilization: float  # percentage
    construction_cost: float  # dollars
    
@dataclass
class BerthConfiguration:
    """Solution from berth segmentation subproblem"""
    num_berths: int
    berth_lengths: List[float]  # meters
    total_length: float  # meters
    flexibility_score: float  # 0-1 scale
    
@dataclass
class AllocationResult:
    """Solution from allocation verification subproblem"""
    feasible: bool
    utilization_rates: Dict[str, float]  # by vessel type
    average_waiting_time: float  # hours
    operational_cost: float  # dollars per year
    
@dataclass
class DivideAndConquerSolution:
    """Integrated solution from all three phases"""
    capacity: CapacitySolution
    configuration: BerthConfiguration
    allocation: AllocationResult
    total_cost: float  # dollars
    computation_time: float  # seconds

In [ ]:
# Create the concrete example from the source
def create_concrete_example():
    """Create the concrete example with vessel data from the source"""
    
    # Define vessel types from source example
    vessel_types = [
        VesselType("Type_A", 180, 15, 8),   # Length=180m, Arrival=15/week, Service=8/week
        VesselType("Type_B", 240, 12, 6),   # Length=240m, Arrival=12/week, Service=6/week
        VesselType("Type_C", 320, 8, 4),    # Length=320m, Arrival=8/week, Service=4/week
        VesselType("Type_D", 420, 4, 3)     # Length=420m, Arrival=4/week, Service=3/week
    ]
    
    # Problem constraints
    max_total_length = 1800  # meters
    min_berth_length = 200   # meters
    construction_cost_per_meter = 25000  # $25,000 per meter
    waiting_cost_per_hour = 8000          # $8,000 per hour
    
    return vessel_types, max_total_length, min_berth_length, construction_cost_per_meter, waiting_cost_per_hour

# Create the example
vessels, max_length, min_berth, construction_cost, waiting_cost = create_concrete_example()

print("Vessel Types:")
for v in vessels:
    print(f"  {v.id}: Length={v.length}m, Arrival={v.arrival_rate}/week, Service={v.service_rate}/week")

print(f"\nProblem Constraints:")
print(f"  Max Total Length: {max_length} meters")
print(f"  Min Berth Length: {min_berth} meters")
print(f"  Construction Cost: ${construction_cost:,} per meter")
print(f"  Waiting Cost: ${waiting_cost:,} per hour")

# Calculate total demand
total_arrival_rate = sum(v.arrival_rate for v in vessels)
total_service_capacity = sum(v.service_rate for v in vessels)
print(f"\nDemand Analysis:")
print(f"  Total Arrival Rate: {total_arrival_rate} vessels/week")
print(f"  Total Service Capacity: {total_service_capacity} vessels/week")

Vessel Types:
  Type_A: Length=180m, Arrival=15/week, Service=8/week
  Type_B: Length=240m, Arrival=12/week, Service=6/week
  Type_C: Length=320m, Arrival=8/week, Service=4/week
  Type_D: Length=420m, Arrival=4/week, Service=3/week

Problem Constraints:
  Max Total Length: 1800 meters
  Min Berth Length: 200 meters
  Construction Cost: $25,000 per meter
  Waiting Cost: $8,000 per hour

Demand Analysis:
  Total Arrival Rate: 39 vessels/week
  Total Service Capacity: 21 vessels/week


In [ ]:
# Phase 1: Capacity Sizing Subproblem
def solve_capacity_sizing(vessel_types: List[VesselType], 
                         max_total_length: float,
                         construction_cost_per_meter: float) -> CapacitySolution:
    """Solve the capacity sizing subproblem to determine total quay length requirements"""
    
    print("=== Phase 1: Capacity Sizing ===")
    
    # Calculate total demand and weighted service requirements
    total_arrival_rate = sum(v.arrival_rate for v in vessel_types)
    weighted_service_rate = sum(v.arrival_rate * v.service_rate for v in vessel_types) / total_arrival_rate
    
    # Target utilization (industry standard: 70-85% for good balance)
    target_utilization = 0.82  # 82% target utilization
    
    # Calculate required number of berths to meet demand
    required_berths = total_arrival_rate / (target_utilization * weighted_service_rate)
    required_berths = max(1, round(required_berths))
    
    print(f"Total arrival rate: {total_arrival_rate:.1f} vessels/week")
    print(f"Weighted service rate: {weighted_service_rate:.1f} vessels/week/berth")
    print(f"Target utilization: {target_utilization*100:.1f}%")
    print(f"Required berths: {required_berths}")
    
    # Calculate total quay length based on vessel size distribution
    # Use weighted average vessel length with safety factor
    avg_vessel_length = sum(v.arrival_rate * v.length for v in vessel_types) / total_arrival_rate
    safety_factor = 1.15  # 15% safety factor for maneuvering and flexibility
    
    # Total quay length = num_berths * (avg_vessel_length * safety_factor)
    total_quay_length = required_berths * (avg_vessel_length * safety_factor)
    
    # Ensure within maximum constraints
    total_quay_length = min(total_quay_length, max_total_length)
    
    # Calculate capacity and utilization
    actual_capacity = required_berths * weighted_service_rate
    actual_utilization = total_arrival_rate / actual_capacity
    
    # Calculate construction cost
    construction_cost = total_quay_length * construction_cost_per_meter
    
    print(f"Average vessel length: {avg_vessel_length:.1f} meters")
    print(f"Total quay length required: {total_quay_length:.1f} meters")
    print(f"Actual capacity: {actual_capacity:.1f} vessels/week")
    print(f"Actual utilization: {actual_utilization*100:.1f}%")
    print(f"Construction cost: ${construction_cost:,.0f}")
    
    return CapacitySolution(
        total_quay_length=total_quay_length,
        total_capacity=actual_capacity,
        utilization=actual_utilization,
        construction_cost=construction_cost
    )

In [ ]:
# Phase 2: Berth Segmentation Subproblem
def solve_berth_segmentation(capacity_solution: CapacitySolution,
                            vessel_types: List[VesselType],
                            min_berth_length: float) -> BerthConfiguration:
    """Solve the berth segmentation subproblem to determine optimal number and sizes of berths"""
    
    print("\n=== Phase 2: Berth Segmentation ===")
    
    total_length = capacity_solution.total_quay_length
    
    # Analyze vessel size distribution to determine optimal segmentation
    vessel_lengths = [v.length for v in vessel_types]
    arrival_rates = [v.arrival_rate for v in vessel_types]
    
    # Sort vessels by length for segmentation analysis
    sorted_data = sorted(zip(vessel_lengths, arrival_rates), key=lambda x: x[0])
    sorted_lengths, sorted_arrivals = zip(*sorted_data)
    
    print(f"Total quay length to segment: {total_length:.1f} meters")
    print(f"Vessel size distribution: {sorted_lengths}")
    print(f"Arrival rate distribution: {sorted_arrivals}")
    
    # Determine optimal number of berths using heuristic
    # Consider both vessel variety and total length constraints
    
    # Calculate length diversity (coefficient of variation)
    mean_length = np.mean(vessel_lengths)
    std_length = np.std(vessel_lengths)
    diversity_factor = std_length / mean_length if mean_length > 0 else 0
    
    print(f"Vessel length diversity (CV): {diversity_factor:.3f}")
    
    # Heuristic: more diverse vessel sizes -> more berths for flexibility
    if diversity_factor > 0.3:
        # High diversity: need more berths for flexibility
        num_berths_options = range(3, 6)
    elif diversity_factor > 0.15:
        # Medium diversity: moderate number of berths
        num_berths_options = range(2, 5)
    else:
        # Low diversity: fewer berths sufficient
        num_berths_options = range(2, 4)
    
    # Evaluate each option
    best_config = None
    best_score = -float('inf')
    
    for num_berths in num_berths_options:
        # Generate berth lengths for this configuration
        berth_lengths = generate_berth_lengths(num_berths, total_length, sorted_lengths, min_berth_length)
        
        if berth_lengths is None:
            continue  # Infeasible configuration
        
        # Calculate flexibility score
        flexibility = calculate_flexibility_score(berth_lengths, sorted_lengths, sorted_arrivals)
        
        # Calculate efficiency score (how well lengths match vessel requirements)
        efficiency = calculate_efficiency_score(berth_lengths, sorted_lengths, sorted_arrivals)
        
        # Combined score (weighted sum)
        score = 0.6 * flexibility + 0.4 * efficiency
        
        print(f"  Option {num_berths} berths: lengths={berth_lengths}, flexibility={flexibility:.3f}, efficiency={efficiency:.3f}, score={score:.3f}")
        
        if score > best_score:
            best_score = score
            best_config = (num_berths, berth_lengths, flexibility)
    
    if best_config is None:
        # Fallback: simple equal division
        num_berths = max(2, int(total_length // 400))  # Default 400m per berth
        berth_lengths = [total_length / num_berths] * num_berths
        flexibility = 0.5
    else:
        num_berths, berth_lengths, flexibility = best_config
    
    print(f"\nOptimal configuration: {num_berths} berths")
    print(f"Berth lengths: {[f'{l:.1f}m' for l in berth_lengths]}")
    print(f"Flexibility score: {flexibility:.3f}")
    
    return BerthConfiguration(
        num_berths=num_berths,
        berth_lengths=berth_lengths,
        total_length=sum(berth_lengths),
        flexibility_score=flexibility
    )

def generate_berth_lengths(num_berths: int, total_length: float, 
                         vessel_lengths: Tuple[float, ...], min_berth_length: float) -> Optional[List[float]]:
    """Generate berth lengths for given number of berths"""
    
    # Simple strategy: create berths that can accommodate different vessel size groups
    if num_berths == 2:
        # Split vessels into two groups
        mid_point = len(vessel_lengths) // 2
        small_vessels = vessel_lengths[:mid_point]
        large_vessels = vessel_lengths[mid_point:]
        
        # Allocate lengths proportionally
        small_ratio = len(small_vessels) / len(vessel_lengths)
        large_ratio = len(large_vessels) / len(vessel_lengths)
        
        berth1_length = max(min_berth_length, total_length * small_ratio)
        berth2_length = total_length - berth1_length
        
        if berth2_length < min_berth_length:
            return None
        
        return [berth1_length, berth2_length]
    
    elif num_berths == 3:
        # Create three berths for small, medium, large vessels
        berth1_length = max(min_berth_length, total_length * 0.3)  # Small vessels
        berth2_length = max(min_berth_length, total_length * 0.35) # Medium vessels
        berth3_length = total_length - berth1_length - berth2_length  # Large vessels
        
        if berth3_length < min_berth_length:
            return None
        
        return [berth1_length, berth2_length, berth3_length]
    
    elif num_berths == 4:
        # Create four berths for more granular segmentation
        base_length = total_length / 4
        berth_lengths = [max(min_berth_length, base_length * factor) for factor in [0.8, 1.0, 1.1, 1.1]]
        
        # Adjust to match total length
        current_total = sum(berth_lengths)
        scale_factor = total_length / current_total
        berth_lengths = [l * scale_factor for l in berth_lengths]
        
        if any(l < min_berth_length for l in berth_lengths):
            return None
        
        return berth_lengths
    
    else:
        # For other numbers, use equal division
        equal_length = total_length / num_berths
        if equal_length < min_berth_length:
            return None
        return [equal_length] * num_berths

def calculate_flexibility_score(berth_lengths: List[float], 
                               vessel_lengths: Tuple[float, ...],
                               arrival_rates: Tuple[float, ...]) -> float:
    """Calculate flexibility score based on how well berths can accommodate different vessels"""
    
    flexibility = 0.0
    total_arrivals = sum(arrival_rates)
    
    for i, vessel_length in enumerate(vessel_lengths):
        vessel_arrival_rate = arrival_rates[i]
        
        # Check how many berths can accommodate this vessel type
        compatible_berths = sum(1 for berth_length in berth_lengths if berth_length >= vessel_length)
        
        # More compatible berths = higher flexibility
        vessel_flexibility = compatible_berths / len(berth_lengths)
        
        # Weight by arrival rate
        flexibility += vessel_flexibility * (vessel_arrival_rate / total_arrivals)
    
    return flexibility

def calculate_efficiency_score(berth_lengths: List[float],
                              vessel_lengths: Tuple[float, ...],
                              arrival_rates: Tuple[float, ...]) -> float:
    """Calculate efficiency score based on space utilization"""
    
    efficiency = 0.0
    total_arrivals = sum(arrival_rates)
    
    for i, vessel_length in enumerate(vessel_lengths):
        vessel_arrival_rate = arrival_rates[i]
        
        # Find best fitting berth (smallest berth that can accommodate vessel)
        fitting_berths = [berth_length for berth_length in berth_lengths if berth_length >= vessel_length]
        
        if fitting_berths:
            best_fit = min(fitting_berths)
            # Efficiency = vessel length / berth length (closer to 1 is better)
            vessel_efficiency = vessel_length / best_fit
        else:
            vessel_efficiency = 0.0  # No berth can accommodate this vessel
        
        efficiency += vessel_efficiency * (vessel_arrival_rate / total_arrivals)
    
    return efficiency

In [ ]:
# Phase 3: Allocation Verification Subproblem
def solve_allocation_verification(configuration: BerthConfiguration,
                                vessel_types: List[VesselType],
                                waiting_cost_per_hour: float) -> AllocationResult:
    """Solve the allocation verification subproblem to validate feasibility"""
    
    print("\n=== Phase 3: Allocation Verification ===")
    
    num_berths = configuration.num_berths
    berth_lengths = configuration.berth_lengths
    
    print(f"Verifying allocation for {num_berths} berths")
    print(f"Berth lengths: {[f'{l:.1f}m' for l in berth_lengths]}")
    
    # Check feasibility: can all vessel types be accommodated?
    feasible = True
    utilization_rates = {}
    
    # Simple assignment: assign each vessel type to the smallest compatible berth
    berth_assignments = {}  # vessel_type_id -> berth_index
    berth_loads = [0.0] * num_berths  # arrival rate assigned to each berth
    
    # Sort vessels by length (smallest first) for efficient assignment
    sorted_vessels = sorted(vessel_types, key=lambda v: v.length)
    
    for vessel in sorted_vessels:
        # Find smallest compatible berth
        compatible_berths = [i for i, length in enumerate(berth_lengths) if length >= vessel.length]
        
        if not compatible_berths:
            print(f"  ❌ No berth can accommodate {vessel.id} ({v.length}m)")
            feasible = False
            break
        
        # Assign to berth with lowest current load
        best_berth = min(compatible_berths, key=lambda i: berth_loads[i])
        berth_assignments[vessel.id] = best_berth
        berth_loads[best_berth] += vessel.arrival_rate
        
        print(f"  {vessel.id} ({v.length}m) -> Berth {best_berth+1} ({berth_lengths[best_berth]:.1f}m)")
    
    if feasible:
        print("  ✅ All vessel types can be accommodated")
        
        # Calculate utilization rates
        for vessel in vessel_types:
            berth_idx = berth_assignments[vessel.id]
            berth_length = berth_lengths[berth_idx]
            
            # Estimate service rate based on berth size
            # Larger berths can handle vessels faster (simplified model)
            size_factor = berth_length / 250.0  # Normalize to 250m reference
            effective_service_rate = vessel.service_rate * min(1.5, size_factor)
            
            utilization = vessel.arrival_rate / effective_service_rate
            utilization_rates[vessel.id] = min(0.95, utilization)  # Cap at 95%
        
        # Calculate average waiting time using queuing approximation
        total_arrival_rate = sum(v.arrival_rate for v in vessel_types)
        avg_utilization = sum(utilization_rates.values()) / len(utilization_rates)
        
        # Simplified waiting time calculation
        if avg_utilization < 0.7:
            avg_waiting_time = 2.0  # hours
        elif avg_utilization < 0.85:
            avg_waiting_time = 4.0  # hours
        elif avg_utilization < 0.95:
            avg_waiting_time = 8.0  # hours
        else:
            avg_waiting_time = 15.0  # hours (high congestion)
        
        # Calculate operational cost (waiting cost per year)
        weeks_per_year = 52
        operational_cost = avg_waiting_time * waiting_cost_per_hour * total_arrival_rate * weeks_per_year
        
        print(f"  Average utilization: {avg_utilization*100:.1f}%")
        print(f"  Average waiting time: {avg_waiting_time:.1f} hours")
        print(f"  Annual operational cost: ${operational_cost:,.0f}")
        
    else:
        # Infeasible solution
        avg_waiting_time = 999.0
        operational_cost = 999999999.0
        for vessel in vessel_types:
            utilization_rates[vessel.id] = 0.0
    
    return AllocationResult(
        feasible=feasible,
        utilization_rates=utilization_rates,
        average_waiting_time=avg_waiting_time,
        operational_cost=operational_cost
    )

In [ ]:
# Main Divide and Conquer Algorithm
def divide_and_conquer_berth_design(vessel_types: List[VesselType],
                                  max_total_length: float,
                                  min_berth_length: float,
                                  construction_cost_per_meter: float,
                                  waiting_cost_per_hour: float) -> DivideAndConquerSolution:
    """Main divide and conquer algorithm for berth design"""
    
    import time
    start_time = time.time()
    
    print("=== Berth Design Divide and Conquer Solution ===")
    print()
    
    # Phase 1: Capacity Sizing
    capacity_solution = solve_capacity_sizing(
        vessel_types, max_total_length, construction_cost_per_meter
    )
    
    # Phase 2: Berth Segmentation
    configuration = solve_berth_segmentation(
        capacity_solution, vessel_types, min_berth_length
    )
    
    # Phase 3: Allocation Verification
    allocation = solve_allocation_verification(
        configuration, vessel_types, waiting_cost_per_hour
    )
    
    # Calculate total cost
    total_cost = capacity_solution.construction_cost + allocation.operational_cost
    
    # Calculate computation time
    computation_time = time.time() - start_time
    
    print("\n=== Final Results ===")
    print(f"Construction cost: ${capacity_solution.construction_cost:,.0f}")
    print(f"Annual operational cost: ${allocation.operational_cost:,.0f}")
    print(f"Total cost (first year): ${total_cost:,.0f}")
    print(f"Average utilization: {sum(allocation.utilization_rates.values())/len(allocation.utilization_rates)*100:.1f}%")
    print(f"Computation time: {computation_time:.4f} seconds")
    print(f"Solution feasible: {'✅ Yes' if allocation.feasible else '❌ No'}")
    
    return DivideAndConquerSolution(
        capacity=capacity_solution,
        configuration=configuration,
        allocation=allocation,
        total_cost=total_cost,
        computation_time=computation_time
    )

In [ ]:
# Solve the concrete example
result = divide_and_conquer_berth_design(
    vessels, max_length, min_berth, construction_cost, waiting_cost
)

print("\n" + "="*60)
print("DETAILED SOLUTION ANALYSIS")
print("="*60)

print("\n📊 CAPACITY ANALYSIS:")
print(f"  Total quay length: {result.capacity.total_quay_length:.1f} meters")
print(f"  Total capacity: {result.capacity.total_capacity:.1f} vessels/week")
print(f"  System utilization: {result.capacity.utilization*100:.1f}%")

print("\n🏗️ BERTH CONFIGURATION:")
print(f"  Number of berths: {result.configuration.num_berths}")
print(f"  Berth lengths: {[f'{l:.1f}m' for l in result.configuration.berth_lengths]}")
print(f"  Flexibility score: {result.configuration.flexibility_score:.3f}")

print("\n⚓ ALLOCATION RESULTS:")
print(f"  Feasibility: {'✅ Feasible' if result.allocation.feasible else '❌ Infeasible'}")
print(f"  Average waiting time: {result.allocation.average_waiting_time:.1f} hours")
print(f"  Annual operational cost: ${result.allocation.operational_cost:,.0f}")

print("\n💰 COST BREAKDOWN:")
print(f"  Construction cost: ${result.capacity.construction_cost:,.0f}")
print(f"  Annual operational cost: ${result.allocation.operational_cost:,.0f}")
print(f"  Total first-year cost: ${result.total_cost:,.0f}")

print("\n⏱️ PERFORMANCE:")
print(f"  Computation time: {result.computation_time:.4f} seconds")
print(f"  Algorithm complexity: O(V log V + V*B)")

if result.allocation.feasible:
    print("\n📈 UTILIZATION BY VESSEL TYPE:")
    for vessel_id, utilization in result.allocation.utilization_rates.items():
        vessel = next(v for v in vessels if v.id == vessel_id)
        print(f"  {vessel_id} ({vessel.length}m): {utilization*100:.1f}%")

=== Berth Design Divide and Conquer Solution ===

=== Phase 1: Capacity Sizing ===
Total arrival rate: 39.0 vessels/week
Weighted service rate: 6.1 vessels/week/berth
Target utilization: 82.0%
Required berths: 8
Average vessel length: 251.8 meters
Total quay length required: 1800.0 meters
Actual capacity: 48.4 vessels/week
Actual utilization: 80.6%
Construction cost: $45,000,000

=== Phase 2: Berth Segmentation ===
Total quay length to segment: 1800.0 meters
Vessel size distribution: (180, 240, 320, 420)
Arrival rate distribution: (15, 12, 8, 4)
Vessel length diversity (CV): 0.310
  Option 3 berths: lengths=[540.0, 630.0, 630.0], flexibility=1.000, efficiency=0.466, score=0.787
  Option 4 berths: lengths=[360.0, 450.0, 495.00000000000006, 495.00000000000006], flexibility=0.974, efficiency=0.675, score=0.855
  Option 5 berths: lengths=[360.0, 360.0, 360.0, 360.0, 360.0], flexibility=0.897, efficiency=0.580, score=0.770

Optimal configuration: 4 berths
Berth lengths: ['360.0m', '450.0m',

In [ ]:
# Visualize the divide and conquer solution
def visualize_divide_conquer_solution(result: DivideAndConquerSolution, 
                                     vessel_types: List[VesselType]) -> None:
    """Create comprehensive visualizations of the divide and conquer solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Berth configuration visualization
    ax1.set_title('Optimal Berth Configuration', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Berth Number')
    ax1.set_ylabel('Berth Length (meters)')
    ax1.grid(True, alpha=0.3)
    
    berth_numbers = list(range(1, result.configuration.num_berths + 1))
    berth_lengths = result.configuration.berth_lengths
    
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#E71D36']
    bars = ax1.bar(berth_numbers, berth_lengths, color=colors[:len(berth_numbers)], alpha=0.8)
    
    # Add value labels and vessel compatibility
    for i, (bar, length) in enumerate(zip(bars, berth_lengths)):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{length:.0f}m', ha='center', va='bottom', fontweight='bold')
        
        # Show compatible vessel types
        compatible = [v.id for v in vessel_types if v.length <= length]
        ax1.text(bar.get_x() + bar.get_width()/2., height/2,
                f'{len(compatible)} types', ha='center', va='center', 
                fontsize=9, rotation=90, color='white', fontweight='bold')
    
    # Plot 2: Cost breakdown
    ax2.set_title('Cost Breakdown Analysis', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Cost ($ millions)')
    ax2.grid(True, alpha=0.3, axis='y')
    
    cost_categories = ['Construction', 'Annual Operational']
    cost_values = [result.capacity.construction_cost/1e6, result.allocation.operational_cost/1e6]
    
    bars = ax2.bar(cost_categories, cost_values, color=['#2E86AB', '#A23B72'], alpha=0.8)
    
    for bar, value in zip(bars, cost_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'${value:.1f}M', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: Vessel allocation and utilization
    ax3.set_title('Vessel Allocation & Utilization', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Vessel Type')
    ax3.set_ylabel('Utilization Rate (%)')
    ax3.grid(True, alpha=0.3)
    
    vessel_ids = list(result.allocation.utilization_rates.keys())
    utilization_rates = [result.allocation.utilization_rates[vid] * 100 for vid in vessel_ids]
    
    bars = ax3.bar(vessel_ids, utilization_rates, color=['#F18F01', '#C73E1D', '#2E86AB', '#A23B72'], alpha=0.8)
    
    # Add target utilization line
    ax3.axhline(y=82, color='red', linestyle='--', 
                label='Target (82%)', alpha=0.7)
    ax3.legend()
    
    for bar, util_rate in zip(bars, utilization_rates):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{util_rate:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Algorithm performance metrics
    ax4.set_title('Algorithm Performance Metrics', fontsize=14, fontweight='bold')
    ax4.set_xlim(0, 3)
    ax4.set_ylim(0, 3)
    ax4.axis('off')
    
    # Create performance summary text
    performance_text = f"""
    🚀 Algorithm Performance
    ══════════════════════════
    
    ⏱️  Computation Time
       {result.computation_time:.4f} seconds
       
    📊 Solution Quality
       Feasible: {'✅' if result.allocation.feasible else '❌'}
       Flexibility: {result.configuration.flexibility_score:.3f}
       
    💰 Cost Efficiency
       Construction: ${result.capacity.construction_cost:,.0f}
       Operational: ${result.allocation.operational_cost:,.0f}/year
       
    ⚓ Capacity Utilization
       Average: {sum(result.allocation.utilization_rates.values())/len(result.allocation.utilization_rates)*100:.1f}%
       Waiting: {result.allocation.average_waiting_time:.1f} hours
       
    🔧 Algorithm Complexity
       O(V log V + V·B)
       V = Vessel types, B = Berths
    """
    
    ax4.text(0.1, 2.5, performance_text, fontsize=10, family='monospace',
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.suptitle('Divide and Conquer Berth Design Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_divide_conquer_solution(result, vessels)

In [ ]:
# Compare with different scenarios and sensitivity analysis
def scenario_analysis():
    """Perform scenario analysis to test algorithm robustness"""
    
    print("=== SCENARIO ANALYSIS ===")
    print()
    
    # Define different scenarios
    scenarios = {
        "Base Case": {
            "vessels": vessels,
            "max_length": 1800,
            "description": "Original problem from source"
        },
        "High Demand": {
            "vessels": [
                VesselType("Type_A", 180, 20, 8),   # +33% demand
                VesselType("Type_B", 240, 16, 6),   # +33% demand
                VesselType("Type_C", 320, 11, 4),   # +38% demand
                VesselType("Type_D", 420, 6, 3)    # +50% demand
            ],
            "max_length": 2000,
            "description": "Increased vessel arrival rates"
        },
        "Large Vessels": {
            "vessels": [
                VesselType("Type_A", 200, 15, 8),   # Larger small vessels
                VesselType("Type_B", 280, 12, 6),   # Larger medium vessels
                VesselType("Type_C", 360, 8, 4),    # Larger large vessels
                VesselType("Type_D", 480, 4, 3)     # Much larger vessels
            ],
            "max_length": 2000,
            "description": "All vessel sizes increased by 10-15%"
        },
        "Constrained Space": {
            "vessels": vessels,
            "max_length": 1400,  # Reduced space
            "description": "Limited quay length availability"
        }
    }
    
    results = {}
    
    for scenario_name, scenario_data in scenarios.items():
        print(f"📋 {scenario_name}:")
        print(f"   {scenario_data['description']}")
        
        # Solve scenario
        scenario_result = divide_and_conquer_berth_design(
            scenario_data["vessels"],
            scenario_data["max_length"],
            min_berth,
            construction_cost,
            waiting_cost
        )
        
        results[scenario_name] = scenario_result
        
        print(f"   ✅ Berths: {scenario_result.configuration.num_berths}")
        print(f"   ✅ Total Length: {scenario_result.capacity.total_quay_length:.0f}m")
        print(f"   ✅ Feasible: {'Yes' if scenario_result.allocation.feasible else 'No'}")
        print(f"   ✅ Total Cost: ${scenario_result.total_cost:,.0f}")
        print(f"   ✅ Compute Time: {scenario_result.computation_time:.4f}s")
        print()
    
    # Create comparison visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    scenario_names = list(results.keys())
    
    # Plot 1: Number of berths comparison
    num_berths = [results[name].configuration.num_berths for name in scenario_names]
    ax1.bar(scenario_names, num_berths, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'], alpha=0.8)
    ax1.set_ylabel('Number of Berths')
    ax1.set_title('Number of Berths by Scenario')
    ax1.grid(True, alpha=0.3, axis='y')
    
    for i, count in enumerate(num_berths):
        ax1.text(i, count + 0.1, str(count), ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Total quay length comparison
    total_lengths = [results[name].capacity.total_quay_length for name in scenario_names]
    ax2.bar(scenario_names, total_lengths, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'], alpha=0.8)
    ax2.set_ylabel('Total Quay Length (meters)')
    ax2.set_title('Total Quay Length by Scenario')
    ax2.grid(True, alpha=0.3, axis='y')
    
    for i, length in enumerate(total_lengths):
        ax2.text(i, length + 10, f'{length:.0f}m', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: Total cost comparison
    total_costs = [results[name].total_cost/1e6 for name in scenario_names]
    ax3.bar(scenario_names, total_costs, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'], alpha=0.8)
    ax3.set_ylabel('Total Cost ($ millions)')
    ax3.set_title('Total Cost by Scenario')
    ax3.grid(True, alpha=0.3, axis='y')
    
    for i, cost in enumerate(total_costs):
        ax3.text(i, cost + 0.5, f'${cost:.1f}M', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Computation time comparison
    compute_times = [results[name].computation_time for name in scenario_names]
    ax4.bar(scenario_names, compute_times, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'], alpha=0.8)
    ax4.set_ylabel('Computation Time (seconds)')
    ax4.set_title('Algorithm Efficiency by Scenario')
    ax4.grid(True, alpha=0.3, axis='y')
    
    for i, time in enumerate(compute_times):
        ax4.text(i, time + 0.0001, f'{time:.4f}s', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Scenario Analysis: Divide and Conquer Algorithm Performance', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return results

# Run scenario analysis
scenario_results = scenario_analysis()

=== SCENARIO ANALYSIS ===

📋 Base Case:
   Original problem from source
=== Berth Design Divide and Conquer Solution ===

=== Phase 1: Capacity Sizing ===
Total arrival rate: 39.0 vessels/week
Weighted service rate: 6.1 vessels/week/berth
Target utilization: 82.0%
Required berths: 8
Average vessel length: 251.8 meters
Total quay length required: 1800.0 meters
Actual capacity: 48.4 vessels/week
Actual utilization: 80.6%
Construction cost: $45,000,000

=== Phase 2: Berth Segmentation ===
Total quay length to segment: 1800.0 meters
Vessel size distribution: (180, 240, 320, 420)
Arrival rate distribution: (15, 12, 8, 4)
Vessel length diversity (CV): 0.310
  Option 3 berths: lengths=[540.0, 630.0, 630.0], flexibility=1.000, efficiency=0.466, score=0.787
  Option 4 berths: lengths=[360.0, 450.0, 495.00000000000006, 495.00000000000006], flexibility=0.974, efficiency=0.675, score=0.855
  Option 5 berths: lengths=[360.0, 360.0, 360.0, 360.0, 360.0], flexibility=0.897, efficiency=0.580, score=0.

### Why this Tier exists vs earlier Tiers
This Tier 2 provides a practical heuristic approach that complements Tier 1's mathematical optimization:
- **Computational efficiency** - O(V log V + VB) complexity vs exponential for robust optimization
- **Scalability** - Handles larger problem instances that are intractable for exact methods
- **Practical implementation** - Easier to understand and implement for port planners
- **Real-time adaptation** - Can be quickly re-run when parameters change
- **Decomposition insight** - Breaks complex problem into manageable subproblems

### Pros / Cons vs Tier 1
**Advantages over Tier 1:**
- **Much faster computation** (milliseconds vs minutes/hours)
- **Scales to larger instances** with many vessel types and berths
- **Intuitive decomposition** that aligns with port planning processes
- **Easy to modify** for specific operational constraints
- **No specialized solver required** - uses basic algorithms only

**Disadvantages vs Tier 1:**
- **No optimality guarantee** - may miss better solutions
- **Limited uncertainty handling** compared to robust optimization
- **Heuristic approximations** in queuing calculations
- **Subproblem coordination** may not be perfectly optimal
- **Local optima risk** in berth segmentation phase

### When to use this Tier
- **Large-scale port planning** where exact optimization is infeasible
- **Preliminary design analysis** to quickly evaluate multiple scenarios
- **Real-time operational planning** requiring fast decisions
- **Resource-constrained environments** without optimization software
- **Stakeholder discussions** where intuitive solutions are preferred
- **Sensitivity analysis** requiring many rapid evaluations
- **Integration with other systems** where modular decomposition is beneficial